# 📝 [제 2편] 수치미분 - 파이썬으로 미분하기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「머신러닝/딥러닝을 위한 수치미분 (II)·(III)」 (ML_LEC_10, ML_LEC_11), 「Python function·lambda」 (ML_LEC_04)

---

## 🎯 학습 목표
1. 수치미분이 무엇인지 알고, 수학 공식으로 하는 미분(해석적 미분)과 비교할 수 있다.
2. 중앙차분 공식을 파이썬 함수로 구현하여 미분계수를 계산할 수 있다.
3. 변수가 여러 개인 함수의 편미분을 수치미분 코드로 계산하고 그 의미를 설명할 수 있다.

---

## 🧭 지난 시간 복습 (제 1편)
* 미분은 **"입력이 아주 조금 변할 때 출력이 얼마나 민감하게 변하는가"** 를 알려 준다.
* f(x) = x² 이면 f'(x) = 2x, 따라서 f'(3) = 6
* 편미분: 관심 있는 변수 하나만 움직이고 **나머지는 상수로 고정**한다.

> **오늘의 질문** 🤔
> 컴퓨터는 "x² 을 미분하면 2x" 라는 공식을 모릅니다. 그렇다면 컴퓨터는 어떻게 미분을 할까요?

[⬅ 이전 : 제 1편 머신러닝을 위한 미분 기초](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/01_derivative.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 3편 벡터와 행렬 - numpy 로 데이터 다루기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/03_vector_matrix.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

# 한글 글꼴 (나눔고딕) 설정
font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 수치미분(Numerical Derivative)이란?

| 구분 | 해석적 미분 (수학 시간) | 수치미분 (컴퓨터) |
|:---:|:---|:---|
| 방법 | 미분 공식을 이용해 **도함수 식**을 구한다 | 아주 작은 Δx 를 직접 넣어 **값을 계산**한다 |
| 예 | f(x) = x² → f'(x) = 2x → f'(3) = 6 | {f(3 + 0.0001) − f(3 − 0.0001)} / 0.0002 = 6.000000000012… |
| 장점 | 정확하다 | 공식을 몰라도 어떤 함수든 계산할 수 있다 |

### 미분계수의 정의와 중앙차분

* 정의:  f'(x) = lim<sub>Δx→0</sub> {f(x + Δx) − f(x)} / Δx
* 컴퓨터는 Δx → 0 (극한)을 할 수 없으므로 **Δx 에 아주 작은 수 (예: 10<sup>−4</sup> = 0.0001)** 를 넣어 계산합니다.
* 이때 x 의 **양쪽**을 똑같이 Δx 만큼 잡는 **중앙차분**을 쓰면 훨씬 정확합니다.

> **중앙차분 공식**
> f'(x) ≒ {f(x + Δx) − f(x − Δx)} / ( 　　　 )

> **[생각해 보기]** 💡 두 점 (x − Δx, f(x − Δx)), (x + Δx, f(x + Δx)) 을 잇는 직선의 기울기를 구해 보면,
> 분모는 두 점의 x 좌표의 차인 (x + Δx) − (x − Δx) = ( 　　　 ) 이 됩니다.

## 📖 개념 다지기 2. 파이썬으로 함수 만들기 : def 와 lambda

수학의 함수 f(x) = x² 을 파이썬에서는 두 가지 방법으로 만들 수 있습니다.

```python
# 방법 1: def
def f(x):
    return x**2

# 방법 2: lambda (한 줄 함수)
f = lambda x : x**2
```

* `lambda 입력 : 식` 은 **"f(x) 를 식으로 바꿔 계산하라"** 는 뜻입니다.
* 수치미분 함수에 **미분할 함수 자체를 입력으로 넘겨줄 때** 아주 편리합니다.

> **[연습 1]** 아래 셀을 실행하고, 빈칸에 결과를 적어 보세요.
> f(3) = ( 　　 ),  g(2) = ( 　　 ),  h(1, 2) = ( 　　 )

In [ ]:
f = lambda x : x**2
g = lambda x : 3*x + 1
h = lambda x, y : 2*x + 3*x*y + y**3

print('f(3)    =', f(3))
print('g(2)    =', g(2))
print('h(1, 2) =', h(1, 2))

## 💻 실습 1. 수치미분 1차 버전 (변수가 1개인 함수)

중앙차분 공식을 그대로 코드로 옮깁니다.

| 수학 | 코드 |
|:---|:---|
| 미분할 함수 f | `f` (def 나 lambda 로 만든 함수를 받는다) |
| 미분계수를 구할 점 x | `x` |
| 아주 작은 수 Δx | `delta_x = 1e-4` (= 0.0001) |
| {f(x + Δx) − f(x − Δx)} / (2Δx) | `(f(x+delta_x) - f(x-delta_x)) / (2*delta_x)` |

In [ ]:
def numerical_derivative(f, x):
    delta_x = 1e-4                                            # 극한(lim) 대신 쓰는 아주 작은 값
    return (f(x + delta_x) - f(x - delta_x)) / (2 * delta_x)  # 중앙차분

# [예제 1] f(x) = x² 에서 f'(3) 구하기
def my_func1(x):
    return x**2

result = numerical_derivative(my_func1, 3)
print("수치미분  f'(3) =", result)
print("공식      f'(3) = 2 × 3 =", 2*3)

> **[확인 1]** 수치미분 결과는 6 과 **정확히 같나요?** 차이가 난다면 그 이유는 무엇일까요?
> (                                                                                   )

### [예제 2] f(x) = 3x·e<sup>x</sup> 에서 f'(2) 구하기

* 수학 공식 (곱의 미분):  f'(x) = 3e<sup>x</sup> + 3x·e<sup>x</sup>
* 따라서  f'(2) = 3e² + 3 × 2 × e² = ( 　　 ) e²  ≒ ( 　　　　 )

In [ ]:
def my_func2(x):
    return 3 * x * np.exp(x)

print("수치미분  f'(2) =", numerical_derivative(my_func2, 2))
print("공식      f'(2) = 3e² + 6e² =", 3*np.exp(2) + 3*2*np.exp(2))

## 🎬 애니메이션 1. Δx 가 작아지면 할선이 접선이 된다

f(x) = x² 위의 점 x = 1 에서
* 🔴 **전진차분**: 두 점 1 과 1 + Δx 를 잇는 직선
* 🟢 **중앙차분**: 두 점 1 − Δx 와 1 + Δx 를 잇는 직선

Δx 가 1.5 에서 0 에 가까워질 때 두 직선의 기울기가 어떻게 변하는지 관찰하세요. (실행 후 ▶ 버튼)

In [ ]:
#@title 🎬 애니메이션 1 실행 (코드 보기: 왼쪽 화살표)
f = lambda x : x**2
x0 = 1.0
hs = np.concatenate([np.linspace(1.5, 0.1, 40), np.full(10, 0.1)])   # Δx 값 변화

fig, ax = plt.subplots(figsize=(8, 5.5))
xs = np.linspace(-1.2, 3.2, 200)

def draw(i):
    h = hs[i]
    ax.clear()
    ax.plot(xs, f(xs), color='royalblue', lw=2, label='f(x) = x²')
    ax.plot(xs, f(x0) + 2*x0*(xs - x0), 'k--', lw=1, alpha=0.5, label="진짜 접선 (기울기 2)")
    s_f = (f(x0 + h) - f(x0)) / h              # 전진차분 기울기
    s_c = (f(x0 + h) - f(x0 - h)) / (2*h)      # 중앙차분 기울기
    ax.plot(xs, f(x0) + s_f*(xs - x0), color='crimson', lw=2, label=f'전진차분 기울기 = {s_f:.3f}')
    ax.plot(xs, f(x0) + s_c*(xs - x0), color='green', lw=2, label=f'중앙차분 기울기 = {s_c:.3f}')
    ax.plot([x0], [f(x0)], 'ko', ms=8, zorder=5)
    ax.plot([x0 + h], [f(x0 + h)], 'o', color='crimson', ms=8, zorder=5)
    ax.plot([x0 - h], [f(x0 - h)], 'o', color='green', ms=8, zorder=5)
    ax.set_xlim(-1.2, 3.2); ax.set_ylim(-2, 9)
    ax.set_title(f'Δx = {h:.2f}  일 때 x = 1 에서의 기울기', fontsize=14)
    ax.legend(loc='upper left'); ax.grid(alpha=0.3)

anim = animation.FuncAnimation(fig, draw, frames=len(hs), interval=150)
plt.close(fig)
show_anim(anim)

> **[관찰 1]**
> ① Δx 가 작아질수록 빨간 직선의 기울기는 ( 　　 ) 에 가까워진다.
> ② 초록 직선(중앙차분)의 기울기는 Δx 의 크기와 상관없이 항상 ( 　　 ) 이다. 왜 그럴까요?
> 　 (힌트: {(1 + Δx)² − (1 − Δx)²} / (2Δx) 을 직접 전개해 보세요.)
> (                                                                                   )

## 🔬 탐구. Δx 는 무조건 작을수록 좋을까?

f(x) = x³ 에서 f'(1) = 3 입니다. Δx 를 10<sup>−1</sup>, 10<sup>−2</sup>, … , 10<sup>−15</sup> 으로 바꿔 가며 **오차 = |수치미분 − 3|** 을 비교해 봅시다.

In [ ]:
f = lambda x : x**3
print(f"{'Δx':>8} | {'전진차분 오차':>14} | {'중앙차분 오차':>14}")
print('-'*44)
hs_list, err_f, err_c = [], [], []
for k in range(1, 16):
    h = 10.0**(-k)
    ef = abs((f(1 + h) - f(1)) / h - 3)
    ec = abs((f(1 + h) - f(1 - h)) / (2*h) - 3)
    hs_list.append(h); err_f.append(ef); err_c.append(ec)
    print(f'{h:8.0e} | {ef:14.2e} | {ec:14.2e}')

plt.figure(figsize=(7, 4.5))
plt.loglog(hs_list, err_f, 'o-', color='crimson', label='전진차분')
plt.loglog(hs_list, err_c, 's-', color='green', label='중앙차분')
plt.gca().invert_xaxis()
plt.xlabel('Δx (오른쪽으로 갈수록 작아짐)'); plt.ylabel('오차 (로그 눈금)')
plt.title("f(x) = x³,  f'(1) 의 오차"); plt.legend(); plt.grid(alpha=0.3, which='both')
plt.show()

> **[탐구 결과]**
> ① 오차가 가장 작아지는 Δx 는 대략 전진차분 ( 　　　 ), 중앙차분 ( 　　　 ) 근처이다.
> ② Δx 가 너무 작아지면(10<sup>−12</sup> 이하) 오차가 오히려 ( 커진다 / 작아진다 ).
>
> 💡 **이유**: 컴퓨터는 소수를 약 16자리까지만 저장합니다. Δx 가 너무 작으면 f(x + Δx) 와 f(x − Δx) 가 거의 같은 수가 되어, 그 차를 계산할 때 **반올림 오차**가 크게 드러납니다.
> 그래서 머신러닝에서는 보통 **Δx = 10<sup>−4</sup>** 정도를 사용합니다.

## 📖 개념 다지기 3. 변수가 여러 개이면? → 변수마다 따로 수치미분

f(x, y) = 2x + 3xy + y³ 에서 점 (1, 2) 에서의 편미분을 구해 봅시다.

**① x 에 대한 편미분** (y = 2 로 고정)
　f(x, 2) = 2x + 6x + 8 = 8x + 8 　→　 ∂f/∂x = ( 　　 )

**② y 에 대한 편미분** (x = 1 로 고정)
　f(1, y) = 2 + 3y + y³ 　→　 ∂f/∂y = 3 + 3y²,　y = 2 를 대입하면 ( 　　 )

> **[의미 해석]** 💡
> * ∂f/∂x = 8 : y 를 2 로 고정한 채 x 를 1 에서 아주 조금 늘리면, f 는 그 변화량의 ( 　 )배만큼 변한다.
> * ∂f/∂y = 15 : x 를 1 로 고정한 채 y 를 2 에서 아주 조금 늘리면, f 는 그 변화량의 ( 　 )배만큼 변한다.
> → 이 점에서 f 는 **( x / y )** 의 변화에 더 민감하다.

컴퓨터도 똑같이 합니다. **변수를 하나씩 번갈아 가며** "그 변수만 ±Δx 만큼 움직이고 나머지는 그대로" 두고 계산합니다.

## 🎬 애니메이션 2. 컴퓨터가 편미분하는 순서 보기

* 왼쪽: f(x, y) = 2x + 3xy + y³ 의 등고선 위의 점 (1, 2)
* 오른쪽: 지금 움직이는 변수 방향으로 자른 단면 곡선과 접선

**1단계**에서는 x 만 움직이고(y 고정), **2단계**에서는 y 만 움직입니다(x 고정).

> 💡 1단계의 단면 f(x, 2) = 8x + 8 은 직선이라 접선과 완전히 겹칩니다. 2단계의 단면은 곡선이라 접선이 한 점에서만 닿습니다.

In [ ]:
#@title 🎬 애니메이션 2 실행
F = lambda x, y : 2*x + 3*x*y + y**3
px, py = 1.0, 2.0
X, Y = np.meshgrid(np.linspace(-0.5, 2.5, 120), np.linspace(0.5, 3.5, 120))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 5))
n = 30
wiggle = 0.6 * np.sin(np.linspace(0, 4*np.pi, n))      # 점이 앞뒤로 움직이는 정도
frames = [('x', w) for w in wiggle] + [('y', w) for w in wiggle]

def draw(i):
    var, w = frames[i]
    axL.clear(); axR.clear()
    cs = axL.contourf(X, Y, F(X, Y), levels=25, cmap='viridis', alpha=0.85)
    axL.plot([px], [py], 'wo', ms=10, mec='k')
    if var == 'x':
        axL.axhline(py, color='orange', lw=2.5, label='y = 2 로 고정 (x 만 이동)')
        axL.plot([px + w], [py], 'o', color='red', ms=10, mec='k')
        t = np.linspace(-0.5, 2.5, 100)
        axR.plot(t, F(t, py), color='orange', lw=2.5, label='단면 f(x, 2) = 8x + 8')
        axR.plot(t, F(px, py) + 8*(t - px), 'r--', lw=1.5, label='접선 기울기 ∂f/∂x = 8')
        axR.plot([px + w], [F(px + w, py)], 'o', color='red', ms=10, mec='k')
        axR.set_xlabel('x'); axR.set_xlim(-0.5, 2.5); axR.set_ylim(0, 30)
        axR.set_title('1단계 : x 에 대한 편미분', fontsize=13)
    else:
        axL.axvline(px, color='magenta', lw=2.5, label='x = 1 로 고정 (y 만 이동)')
        axL.plot([px], [py + w], 'o', color='red', ms=10, mec='k')
        t = np.linspace(0.5, 3.5, 100)
        axR.plot(t, F(px, t), color='magenta', lw=2.5, label='단면 f(1, y) = 2 + 3y + y³')
        axR.plot(t, F(px, py) + 15*(t - py), 'r--', lw=1.5, label='접선 기울기 ∂f/∂y = 15')
        axR.plot([py + w], [F(px, py + w)], 'o', color='red', ms=10, mec='k')
        axR.set_xlabel('y'); axR.set_xlim(0.5, 3.5); axR.set_ylim(0, 60)
        axR.set_title('2단계 : y 에 대한 편미분', fontsize=13)
    axL.set_xlabel('x'); axL.set_ylabel('y'); axL.set_title('f(x, y) 등고선과 점 (1, 2)', fontsize=13)
    axL.legend(loc='lower right'); axR.legend(loc='upper left'); axR.grid(alpha=0.3)

anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
show_anim(anim)

## 💻 실습 2. 수치미분 최종 버전 (변수가 여러 개인 함수)

변수가 여러 개이면 입력을 **numpy 배열** 하나에 모아서 넘겨줍니다. 예: (x, y) = (1, 2) → `np.array([1.0, 2.0])`

코드의 핵심 아이디어 ①~④

| 번호 | 하는 일 |
|:---:|:---|
| ① | 계산한 편미분 값을 저장할 빈 배열 `grad` 를 입력과 같은 모양으로 만든다 |
| ② | `np.nditer` 로 변수를 **하나씩 차례대로** 가리킨다 (iterator) |
| ③ | 가리킨 변수 하나만 +Δx, −Δx 로 바꿔서 f 를 계산한다 (나머지 변수는 그대로 = 상수 취급) |
| ④ | 바꿨던 변수를 **원래 값으로 되돌린다** (numpy 배열은 함수 안에서 바꾸면 원본도 바뀌기 때문) |

아래 코드는 계산 과정을 보여 주는 `debug` 출력이 들어간 버전입니다.

In [ ]:
def numerical_derivative(f, x, debug=False):
    delta_x = 1e-4
    grad = np.zeros_like(x)                                              # ① 결과 저장

    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])     # ② 변수 하나씩
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]                                                 #    원래 값 보관

        x[idx] = float(tmp_val) + delta_x                                # ③ 이 변수만 +Δx
        fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x                                #    이 변수만 −Δx
        fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)

        if debug:
            print(f'변수 위치 {idx} : 원래 값 = {tmp_val}, 편미분 값 = {grad[idx]:.6f}')

        x[idx] = tmp_val                                                 # ④ 원래 값으로 복원
        it.iternext()
    return grad

### [예제 3] 1변수 함수 f(x) = x², f'(3)
입력이 하나여도 배열 `np.array([3.0])` 로 넘겨야 합니다.

In [ ]:
def func1(input_obj):
    x = input_obj[0]
    return x**2

print(numerical_derivative(func1, np.array([3.0]), debug=True))

### [예제 4] 2변수 함수 f(x, y) = 2x + 3xy + y³, 점 (1.0, 2.0)
⚠️ 입력은 반드시 **실수(1.0, 2.0)** 로 넣으세요. 정수(1, 2)로 넣으면 0.0001 을 더해도 정수로 잘려 계산이 틀어집니다.

In [ ]:
def func2(input_obj):
    x = input_obj[0]
    y = input_obj[1]
    return 2*x + 3*x*y + np.power(y, 3)

grad = numerical_derivative(func2, np.array([1.0, 2.0]), debug=True)
print('결과 (∂f/∂x, ∂f/∂y) =', grad)

> **[확인 2]** 개념 다지기 3 에서 손으로 구한 값 ( 　 , 　 ) 과 일치하나요? ( 예 / 아니오 )

### [예제 5] 4변수 함수 f(w, x, y, z) = wx + xyz + 3w + zy², 점 (w, x, y, z) = (1, 2, 3, 4)
변수 4개를 2 × 2 행렬로 묶어서 넘겨줍니다.  →  [[w, x], [y, z]] = [[1, 2], [3, 4]]

손으로 먼저 구해 봅시다.
* ∂f/∂w = x + 3 = ( 　 )
* ∂f/∂x = w + yz = ( 　 )
* ∂f/∂y = xz + 2zy = ( 　 )
* ∂f/∂z = xy + y² = ( 　 )

In [ ]:
def func4(input_obj):
    w = input_obj[0, 0]
    x = input_obj[0, 1]
    y = input_obj[1, 0]
    z = input_obj[1, 1]
    return w*x + x*y*z + 3*w + z*np.power(y, 2)

inp = np.array([[1.0, 2.0], [3.0, 4.0]])
print(numerical_derivative(func4, inp, debug=True))

> **[생각해 보기]** 💡 결과도 입력과 똑같이 **2 × 2 행렬** 모양으로 나옵니다.
> 나중에 딥러닝에서 수백 개의 가중치를 행렬 W 에 담아 한꺼번에 미분할 때 바로 이 코드를 사용합니다!

## ✏️ 연습 문제

> **[문제 1]** f(x) = ln x + 1/x 일 때, f'(2) 를 손으로 구하고 수치미분으로 확인하세요.
> 손 계산: f'(x) = ( 　　　　　 ),　f'(2) = ( 　　 )

> **[문제 2]** f(x, y) = x²y + e<sup>y</sup> 일 때, 점 (1, 0) 에서 ∂f/∂x, ∂f/∂y 를 손으로 구하고 수치미분으로 확인하세요.
> 손 계산: ∂f/∂x = ( 　　 ),　∂f/∂y = ( 　　 )

아래 셀의 `______` 부분을 채워 실행하세요.

In [ ]:
# [문제 1]
p1 = lambda v : np.log(v[0]) + 1/v[0]
print('문제 1 :', numerical_derivative(p1, np.array([______])))

# [문제 2]
def p2(v):
    x = v[0]
    y = v[1]
    return ______

print('문제 2 :', numerical_derivative(p2, np.array([______, ______])))

In [ ]:
#@title 🔑 정답 확인 (먼저 풀어 본 뒤에 실행하세요) { display-mode: "form" }
_p1 = lambda v : np.log(v[0]) + 1/v[0]
_p2 = lambda v : v[0]**2 * v[1] + np.exp(v[1])
print("문제 1 : f'(x) = 1/x − 1/x²,  f'(2) = 1/2 − 1/4 = 0.25")
print('   수치미분 →', numerical_derivative(_p1, np.array([2.0])))
print("문제 2 : ∂f/∂x = 2xy = 0,  ∂f/∂y = x² + e^y = 1 + 1 = 2")
print('   수치미분 →', numerical_derivative(_p2, np.array([1.0, 0.0])))

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 수치미분 | 공식 없이 작은 Δx 를 직접 넣어 미분계수를 **계산**하는 방법 |
| 중앙차분 | f'(x) ≒ {f(x + Δx) − f(x − Δx)} / (2Δx),  Δx = 10<sup>−4</sup> |
| 다변수 | 변수를 하나씩 번갈아 움직이며 계산 (나머지는 고정 = 편미분) |
| 주의 | Δx 가 너무 작으면 반올림 오차, 입력은 실수형 numpy 배열 |

---

### 🏆 도전 과제
f(W) = (W − 3)² 에서 W = 0 일 때 수치미분 값은 ( 　　 ) 입니다.
이 값이 **음수**라는 것은 W 를 ( 늘려야 / 줄여야 ) f(W) 가 작아진다는 뜻입니다.
→ 다음 시간에는 이 아이디어로 인공지능이 **오차를 줄여 가는 방법(경사하강법)** 을 배웁니다!

(                                                                                                                            )

---

[⬅ 이전 : 제 1편 머신러닝을 위한 미분 기초](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/01_derivative.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 3편 벡터와 행렬 - numpy 로 데이터 다루기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/03_vector_matrix.ipynb)